# ETL Workshop 2: recorrido de datos

Este notebook documenta el proceso que llevó a las decisiones de limpieza, transformación y matching entre Spotify y Grammys.

Objetivos del notebook:
- Cargar los dos CSV en pandas.
- Ver los `DataFrame` claramente con `head()`, `info()` y `describe()`.
- Revisar nulos, duplicados y calidad general de los datos.
- Entender por qué se tomaron decisiones de cleaning y cómo se preservaron los géneros sin duplicar métricas.
- Explorar cómo se hace el cruce entre ambos datasets cuando no existe una llave compartida.
- Dejar planteadas las preguntas analíticas que puede responder el warehouse final.


## 0. Rol de Airflow dentro del workshop

Este notebook **no reemplaza** el ETL orquestado. Su función es documentar el profiling, justificar decisiones de calidad y dejar evidencia del razonamiento analítico.

El uso de Apache Airflow en el proyecto ocurre en el DAG `dags/spotify_grammy_etl.py`, donde se orquestan estas tareas:
- `bootstrap_grammy_source_db`: crea y carga la base fuente de Grammys.
- `extract_spotify_csv`: lee Spotify desde CSV.
- `extract_grammy_db`: lee Grammys desde base de datos.
- `clean_datasets`: ejecuta validaciones y cleaning.
- `transform_and_merge`: hace el cruce y el enriquecimiento.
- `export_csv_to_local_drive`: exporta el CSV final.
- `load_data_warehouse`: carga el Data Warehouse.

El script `scripts/run_pipeline.py` existe solo como ayuda local para ejecutar la misma lógica sin levantar Airflow, pero la orquestación formal del workshop sí está en Apache Airflow.


## 1. Configuración y rutas

El notebook busca primero los archivos en `data/raw/` y, si no existen ahí, usa los archivos de `~/Downloads/`.

In [1]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

spotify_candidates = [
    PROJECT_ROOT / 'data' / 'raw' / 'spotify_dataset.csv',
    Path.home() / 'Downloads' / 'spotify_dataset.csv',
]
grammy_candidates = [
    PROJECT_ROOT / 'data' / 'raw' / 'the_grammy_awards.csv',
    Path.home() / 'Downloads' / 'the_grammy_awards.csv',
]

spotify_path = next(path for path in spotify_candidates if path.exists())
grammy_path = next(path for path in grammy_candidates if path.exists())

print('Project root:', PROJECT_ROOT)
print('Spotify path:', spotify_path)
print('Grammy path:', grammy_path)
plt.style.use("ggplot")


Project root: /Users/andresfelipecastrosalazar/Desktop/UAO/ETL/etl-workshop-2
Spotify path: /Users/andresfelipecastrosalazar/Downloads/spotify_dataset.csv
Grammy path: /Users/andresfelipecastrosalazar/Downloads/the_grammy_awards.csv


## 2. Carga inicial de los CSV en pandas

In [2]:
spotify_df = pd.read_csv(spotify_path)
grammy_df = pd.read_csv(grammy_path)

print('Spotify shape:', spotify_df.shape)
print('Grammy shape:', grammy_df.shape)

display(spotify_df.head())
display(grammy_df.head())

Spotify shape: (114000, 21)
Grammy shape: (4810, 10)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Soundtrack),Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


,year,title,published_at,updated_at,category,nominee,artist,workers,img,winner
0,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,Bad Guy,Billie Eilish,"Finneas O'Connell, producer; Rob Kinelski & Finneas O'Connell, engineers/mixers; John Greenham, mastering engineer",https://www.grammy.com/sites/com/files/styles/artist_circle/public/muzooka/Billie%2BEilish/Billie%2520Eilish_1_1_159...,True
1,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,"Hey, Ma",Bon Iver,"BJ Burton, Brad Cook, Chris Messina & Justin Vernon, producers; BJ Burton, Zach Hanson & Chris Messina, engineers/mi...",https://www.grammy.com/sites/com/files/styles/artist_circle/public/muzooka/Bon%2BIver/Bon%2520Iver_1_1_1578385181.jp...,True
2,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,7 rings,Ariana Grande,"Charles Anderson, Tommy Brown, Michael Foster & Victoria Monet, producers; Serban Ghenea, John Hanes, Billy Hickey &...",https://www.grammy.com/sites/com/files/styles/artist_circle/public/muzooka/Ariana%2BGrande/Ariana%2520Grande_1_1_157...,True
3,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,Hard Place,H.E.R.,"Rodney “Darkchild” Jerkins, producer; Joseph Hurtado, Jaycen Joshua, Derek Keota & Miki Tsutsumi, engineers/mixers; ...",https://www.grammy.com/sites/com/files/styles/artist_circle/public/muzooka/H.E.R./H.E.R._1_1_1594631035.jpg?itok=ClJ...,True
4,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,Talk,Khalid,"Disclosure & Denis Kosiak, producers; Ingmar Carlson, Jon Castelli, Josh Deguzman, John Kercy, Denis Kosiak, Guy Law...",https://www.grammy.com/sites/com/files/styles/artist_circle/public/muzooka/Khalid/Khalid_1_1_1594578772.jpg?itok=2Hx...,True


## 3. Estructura general de los DataFrames

Aquí vemos columnas, tipos de datos y primeras pistas de calidad.

In [3]:
print('Spotify columns:')
print(list(spotify_df.columns))
print('\nGrammy columns:')
print(list(grammy_df.columns))

print('\nSpotify info')
spotify_df.info()

print('\nGrammy info')
grammy_df.info()

Spotify columns:
['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

Grammy columns:
['year', 'title', 'published_at', 'updated_at', 'category', 'nominee', 'artist', 'workers', 'img', 'winner']

Spotify info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          1

## 4. Describe de los DataFrames

Uso `describe(include='all')` para observar tanto variables numéricas como categóricas.

## 4.1 Variables clave identificadas

Antes de limpiar, conviene dejar explícitas las variables clave para integración y análisis.


In [ ]:
key_variables = pd.DataFrame([
    {'source': 'Spotify', 'variable': 'track_id', 'role': 'identificador canónico del track', 'why_it_matters': 'permite consolidar duplicados sin deduplicar por nombre'},
    {'source': 'Spotify', 'variable': 'track_name', 'role': 'nombre del track', 'why_it_matters': 'se usa en el matching con nominee'},
    {'source': 'Spotify', 'variable': 'artists', 'role': 'artistas asociados', 'why_it_matters': 'de aquí se deriva primary_artist para el join'},
    {'source': 'Spotify', 'variable': 'track_genre', 'role': 'género reportado', 'why_it_matters': 'sirve para perfilamiento analítico, no como llave de join'},
    {'source': 'Spotify', 'variable': 'popularity', 'role': 'métrica de popularidad', 'why_it_matters': 'sirve para contrastar si popularidad se refleja en historial Grammy'},
    {'source': 'Spotify', 'variable': 'danceability / energy / valence / instrumentalness', 'role': 'atributos de audio', 'why_it_matters': 'permiten perfilar características de tracks conectados con Grammys'},
    {'source': 'Grammys', 'variable': 'artist', 'role': 'artista del premio', 'why_it_matters': 'se usa en el matching textual con primary_artist'},
    {'source': 'Grammys', 'variable': 'nominee', 'role': 'canción o nominado', 'why_it_matters': 'se usa en el matching textual con track_name'},
    {'source': 'Grammys', 'variable': 'category', 'role': 'categoría del premio', 'why_it_matters': 'sirve para segmentar el tipo de reconocimiento'},
    {'source': 'Grammys', 'variable': 'year', 'role': 'año del premio', 'why_it_matters': 'permite análisis temporal'},
    {'source': 'Grammys', 'variable': 'winner', 'role': 'bandera de ganador', 'why_it_matters': 'en este dataset se comporta como histórico de ganadores'},
])
display(key_variables)


In [4]:
spotify_describe = spotify_df.describe(include='all').T
grammy_describe = grammy_df.describe(include='all').T

display(spotify_describe)
display(grammy_describe)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,114000.0,NaN,NaN,NaN,56999.5,32909.109681,0.0,28499.75,56999.5,85499.25,113999.0
track_id,114000,89741,6S3JlDAGk3uu3NtZbPnuhS,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
artists,113999,31437,The Beatles,279,NaN,NaN,NaN,NaN,NaN,NaN,NaN
album_name,113999,46589,Alternative Christmas 2022,195,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track_name,113999,73608,Run Rudolph Run,151,NaN,NaN,NaN,NaN,NaN,NaN,NaN
popularity,114000.0,NaN,NaN,NaN,33.238535,22.305078,0.0,17.0,35.0,50.0,100.0
duration_ms,114000.0,NaN,NaN,NaN,228029.153114,107297.712645,0.0,174066.0,212906.0,261506.0,5237295.0
explicit,114000,2,False,104253,NaN,NaN,NaN,NaN,NaN,NaN,NaN
danceability,114000.0,NaN,NaN,NaN,0.5668,0.173542,0.0,0.456,0.58,0.695,0.985
energy,114000.0,NaN,NaN,NaN,0.641383,0.251529,0.0,0.472,0.685,0.854,1.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
year,4810.0,NaN,NaN,NaN,1995.566944,17.14972,1958.0,1983.0,1998.0,2010.0,2019.0
title,4810,62,62nd Annual GRAMMY Awards (2019),433,NaN,NaN,NaN,NaN,NaN,NaN,NaN
published_at,4810,4,2017-11-28T00:03:45-08:00,4205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
updated_at,4810,10,2019-09-10T01:08:19-07:00,778,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,4810,638,Song Of The Year,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nominee,4804,4131,Bridge Over Troubled Water,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
artist,2970,1658,(Various Artists),66,NaN,NaN,NaN,NaN,NaN,NaN,NaN
workers,2620,2366,"John Williams, composer (John Williams)",20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
img,3443,1463,https://www.grammy.com/sites/com/files/styles/artist_circle/public/muzooka/John%2BWilliams/John%2520Williams_1_1_159...,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
winner,4810,1,True,4810,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Nulos, valores faltantes y duplicados

Esta parte es clave porque aquí aparecen varias de las decisiones de cleaning.

In [5]:
spotify_missing = spotify_df.isna().sum().sort_values(ascending=False).rename('missing_count').to_frame()
spotify_missing['missing_pct'] = (spotify_missing['missing_count'] / len(spotify_df)).round(4)

grammy_missing = grammy_df.isna().sum().sort_values(ascending=False).rename('missing_count').to_frame()
grammy_missing['missing_pct'] = (grammy_missing['missing_count'] / len(grammy_df)).round(4)

spotify_dup_track_ids = spotify_df['track_id'].duplicated().sum()
spotify_unique_track_ids = spotify_df['track_id'].nunique(dropna=True)
spotify_blank_first_col = spotify_df.columns[0]

grammy_winner_values = grammy_df['winner'].astype(str).value_counts(dropna=False)
grammy_missing_artist = grammy_df['artist'].isna().sum() + (grammy_df['artist'].fillna('').str.strip() == '').sum()

print('Spotify duplicate track_id rows:', spotify_dup_track_ids)
print('Spotify unique track_id:', spotify_unique_track_ids)
print('Spotify first column name:', repr(spotify_blank_first_col))
print('Grammy winner values:')
display(grammy_winner_values.to_frame('count'))
print('Grammy rows with missing/blank artist:', grammy_missing_artist)

display(spotify_missing)
display(grammy_missing)

Spotify duplicate track_id rows: 24259
Spotify unique track_id: 89741
Spotify first column name: 'Unnamed: 0'
Grammy winner values:


,count
winner,
True,4810


Grammy rows with missing/blank artist: 3680


,missing_count,missing_pct
artists,1,0.0
album_name,1,0.0
track_name,1,0.0
Unnamed: 0,0,0.0
mode,0,0.0
time_signature,0,0.0
tempo,0,0.0
valence,0,0.0
liveness,0,0.0
instrumentalness,0,0.0


,missing_count,missing_pct
workers,2190,0.4553
artist,1840,0.3825
img,1367,0.2842
nominee,6,0.0012
year,0,0.0000
title,0,0.0000
published_at,0,0.0000
updated_at,0,0.0000
category,0,0.0000
winner,0,0.0000


## 5.1 Revisión de outliers numéricos

Para el profiling también revisamos variables numéricas con una regla simple de IQR. Esto no significa que todos los outliers se eliminen, pero sí nos ayuda a detectar valores extremos que merecen explicación.


In [ ]:
spotify_profile = spotify_df.copy().rename(columns={spotify_df.columns[0]: 'source_row_id'})
profile_numeric_cols = ['popularity', 'duration_ms', 'danceability', 'energy', 'instrumentalness', 'valence', 'tempo']
for col in profile_numeric_cols:
    spotify_profile[col] = pd.to_numeric(spotify_profile[col], errors='coerce')

def iqr_outlier_summary(df, columns):
    rows = []
    for col in columns:
        series = df[col].dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (df[col] < lower) | (df[col] > upper)
        rows.append({
            'variable': col,
            'q1': round(float(q1), 4),
            'q3': round(float(q3), 4),
            'iqr': round(float(iqr), 4),
            'lower_bound': round(float(lower), 4),
            'upper_bound': round(float(upper), 4),
            'outlier_count': int(mask.sum()),
            'outlier_pct': round(float(mask.mean()), 4),
        })
    return pd.DataFrame(rows)

spotify_outlier_summary = iqr_outlier_summary(spotify_profile, profile_numeric_cols)
display(spotify_outlier_summary)

for col in ['popularity', 'duration_ms', 'tempo']:
    bounds = spotify_outlier_summary.set_index('variable').loc[col]
    sample = spotify_profile.loc[
        (spotify_profile[col] < bounds['lower_bound']) | (spotify_profile[col] > bounds['upper_bound']),
        ['track_id', 'track_name', 'artists', col]
    ].head(10)
    print(f'Outliers de ejemplo para {col}:')
    display(sample)


## 5.2 Estadísticas básicas y gráficas exploratorias

Además de `describe`, conviene dejar algunas visualizaciones simples para entender distribuciones y categorías dominantes.


In [ ]:
spotify_plot = spotify_profile.copy()
spotify_plot['track_genre'] = spotify_plot['track_genre'].fillna('').astype(str).str.strip()
grammy_plot = grammy_df.copy()
grammy_plot['category'] = grammy_plot['category'].fillna('').astype(str).str.strip()
grammy_plot['year'] = pd.to_numeric(grammy_plot['year'], errors='coerce')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
spotify_plot['popularity'].dropna().plot(kind='hist', bins=30, ax=axes[0, 0], color='#ba4a00', edgecolor='white')
axes[0, 0].set_title('Distribución de popularity en Spotify')
axes[0, 0].set_xlabel('popularity')

spotify_plot['danceability'].dropna().plot(kind='hist', bins=30, ax=axes[0, 1], color='#1f77b4', edgecolor='white')
axes[0, 1].set_title('Distribución de danceability')
axes[0, 1].set_xlabel('danceability')

spotify_plot['track_genre'].value_counts().head(10).sort_values().plot(kind='barh', ax=axes[1, 0], color='#2a9d8f')
axes[1, 0].set_title('Top 10 géneros en Spotify')
axes[1, 0].set_xlabel('conteo')

grammy_plot.loc[grammy_plot['category'] != '', 'category'].value_counts().head(10).sort_values().plot(kind='barh', ax=axes[1, 1], color='#6a4c93')
axes[1, 1].set_title('Top 10 categorías Grammy')
axes[1, 1].set_xlabel('conteo')

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
grammy_plot['year'].dropna().astype(int).value_counts().sort_index().plot(ax=axes[0], color='#264653')
axes[0].set_title('Registros Grammy por año')
axes[0].set_xlabel('año')
axes[0].set_ylabel('conteo')

spotify_plot['instrumentalness'].dropna().plot(kind='hist', bins=30, ax=axes[1], color='#e76f51', edgecolor='white')
axes[1].set_title('Distribución de instrumentalness')
axes[1].set_xlabel('instrumentalness')

plt.tight_layout()
plt.show()

display(spotify_plot['track_genre'].value_counts().head(10).rename('count').to_frame())
display(grammy_plot.loc[grammy_plot['category'] != '', 'category'].value_counts().head(10).rename('count').to_frame())


## 6. Decisiones de cleaning derivadas del profiling

A partir de lo anterior, las decisiones tomadas fueron:

1. **Spotify trae una columna inicial vacía**. No es una variable de negocio; se usa solo como identificador de fila de origen.
2. **Spotify tiene muchos `track_id` duplicados**. El profiling mostró que no representan canciones distintas: casi siempre son la misma canción repetida por múltiples géneros.
3. **Por eso no conviene escoger un solo género ganador**. La decisión correcta es construir una fila canónica por `track_id`, guardar los géneros agregados en `track_genres` y, en el ETL completo, preservarlos además en `bridge_track_genre`.
4. **Los conflictos de `popularity` también deben quedar trazables**. En vez de esconderlos, se guarda una popularidad canónica y además `popularity_min`, `popularity_max` y un flag de conflicto.
5. **Los tipos numéricos deben normalizarse explícitamente** para evitar problemas posteriores en cálculos y carga al warehouse.
6. **Se necesita un `primary_artist`** porque Spotify puede traer varios artistas concatenados y el cruce con Grammys funciona mejor con una clave principal.
7. **El merge requiere texto normalizado**. No existe una llave compartida entre Spotify y Grammys, así que el match más fuerte sale de `primary_artist + track_name` contra `artist + nominee`, todos normalizados.
8. **Los géneros no participan en el join principal**. Se preservan para análisis posteriores, no como llave de integración entre fuentes.
9. **En Grammys el campo `winner` aparece siempre como `True`**, así que se interpreta como una historia de ganadores más que una tabla completa de nominados con ganadores y perdedores.
10. **Grammys tiene muchos artistas faltantes**. Esos registros se conservan para contexto histórico, pero no sirven para un join exacto por artista.


## 7. Helpers de normalización

Estas funciones replican la lógica conceptual usada en el pipeline.

In [6]:
def normalize_text(value: str) -> str:
    value = '' if pd.isna(value) else str(value)
    value = value.strip().lower()
    value = re.sub(r'\([^)]*\)', ' ', value)
    value = re.sub(r'\b(feat|featuring|ft)\b\.?', ' ', value)
    value = re.sub(r'[^a-z0-9]+', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

def primary_artist(value: str) -> str:
    value = '' if pd.isna(value) else str(value)
    tokens = re.split(r'\s*(?:;|,|&|\bx\b|\bfeat\b\.?|\bfeaturing\b|\bft\b\.?)\s*', value, maxsplit=1, flags=re.IGNORECASE)
    return tokens[0].strip() if tokens else value.strip()

## 8. Cleaning de Spotify paso a paso

In [ ]:
spotify_work = spotify_df.copy()

# La primera columna viene sin nombre; la renombramos para dejar trazabilidad.
spotify_work = spotify_work.rename(columns={spotify_work.columns[0]: 'source_row_id'})

duplicate_summary = (
    spotify_work.groupby('track_id', dropna=False)
    .agg(
        rows=('track_id', 'size'),
        distinct_artists=('artists', 'nunique'),
        distinct_track_names=('track_name', 'nunique'),
        distinct_album_names=('album_name', 'nunique'),
        distinct_genres=('track_genre', 'nunique'),
        distinct_popularity=('popularity', 'nunique')
    )
    .reset_index()
)

numeric_cols = [
    'popularity', 'duration_ms', 'danceability', 'energy', 'key', 'loudness',
    'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'time_signature'
]

spotify_work['track_id'] = spotify_work['track_id'].fillna('').astype(str).str.strip()
spotify_work['track_name'] = spotify_work['track_name'].fillna('').astype(str).str.strip()
spotify_work['album_name'] = spotify_work['album_name'].fillna('').astype(str).str.strip()
spotify_work['artists'] = spotify_work['artists'].fillna('').astype(str).str.strip()
spotify_work['track_genre'] = spotify_work['track_genre'].fillna('').astype(str).str.lower().str.strip()

spotify_work['explicit'] = (
    spotify_work['explicit']
    .astype(str)
    .str.lower()
    .map({'true': 1, 'false': 0})
    .fillna(0)
    .astype(int)
)

for col in numeric_cols:
    spotify_work[col] = pd.to_numeric(spotify_work[col], errors='coerce')

spotify_work['primary_artist'] = spotify_work['artists'].apply(primary_artist).replace('', pd.NA).fillna('Unknown Artist')
spotify_work['primary_artist_normalized'] = spotify_work['primary_artist'].apply(normalize_text)
spotify_work['track_name_normalized'] = spotify_work['track_name'].apply(normalize_text)
spotify_work['album_name_normalized'] = spotify_work['album_name'].apply(normalize_text)
spotify_work['duration_minutes'] = (spotify_work['duration_ms'] / 60000).round(2)

spotify_before = len(spotify_work)
spotify_invalid = ((spotify_work['track_id'] == '') | (spotify_work['track_name'] == '')).sum()
spotify_work = spotify_work[(spotify_work['track_id'] != '') & (spotify_work['track_name'] != '')].copy()

spotify_grouped = []
for track_id, group in spotify_work.groupby('track_id', sort=False):
    genres = sorted({genre for genre in group['track_genre'].dropna().astype(str) if genre})
    popularity_counts = group['popularity'].value_counts(dropna=True).sort_index()
    canonical_popularity = popularity_counts.sort_values(ascending=False)
    max_frequency = canonical_popularity.max()
    canonical_popularity = canonical_popularity[canonical_popularity == max_frequency].index.max()
    representative = group[group['popularity'] == canonical_popularity].iloc[0].copy()
    representative['track_genres'] = ' | '.join(genres)
    representative['genre_count'] = len(genres)
    representative['source_row_count'] = len(group)
    representative['popularity_min'] = int(group['popularity'].min())
    representative['popularity_max'] = int(group['popularity'].max())
    representative['popularity_conflict_flag'] = int(group['popularity'].nunique() > 1)
    representative['popularity_values_observed'] = ' | '.join(str(int(value)) for value in sorted(group['popularity'].dropna().unique()))
    spotify_grouped.append(representative)

spotify_clean = pd.DataFrame(spotify_grouped).sort_values(['primary_artist', 'track_name']).reset_index(drop=True)

spotify_metrics = pd.DataFrame([
    {'metric': 'raw_rows', 'value': spotify_before},
    {'metric': 'rows_missing_track_id_or_track_name', 'value': int(spotify_invalid)},
    {'metric': 'track_id_groups_collapsed', 'value': int(len(spotify_work) - len(spotify_clean))},
    {'metric': 'multi_genre_track_ids', 'value': int((spotify_clean['genre_count'] > 1).sum())},
    {'metric': 'popularity_conflict_track_ids', 'value': int(spotify_clean['popularity_conflict_flag'].sum())},
    {'metric': 'rows_after_canonical_track_build', 'value': len(spotify_clean)},
])

display(spotify_metrics)
display(spotify_clean[['track_id','primary_artist','track_name','track_genres','genre_count','popularity','popularity_min','popularity_max','popularity_conflict_flag']].head())

## 8.1 Verificación puntual de `track_id` duplicados

Aquí validamos la hipótesis más importante del profiling: que los duplicados de `track_id` no son canciones distintas, sino la misma canción repetida principalmente por género.


In [ ]:
duplicate_groups = duplicate_summary[duplicate_summary['rows'] > 1].sort_values(
    ['rows', 'distinct_genres', 'distinct_popularity'],
    ascending=[False, False, False]
)

multi_genre_groups = duplicate_groups[
    (duplicate_groups['distinct_artists'] == 1)
    & (duplicate_groups['distinct_track_names'] == 1)
    & (duplicate_groups['distinct_album_names'] == 1)
    & (duplicate_groups['distinct_genres'] > 1)
]

popularity_conflict_groups = duplicate_groups[duplicate_groups['distinct_popularity'] > 1]

display(duplicate_groups.head(10))
display(multi_genre_groups.head(10))
display(popularity_conflict_groups.head(10))

if not multi_genre_groups.empty:
    sample_track_id = multi_genre_groups.iloc[0]['track_id']
else:
    sample_track_id = duplicate_groups.iloc[0]['track_id']

display(
    spotify_work.loc[
        spotify_work['track_id'] == sample_track_id,
        ['track_id', 'track_name', 'artists', 'album_name', 'track_genre', 'popularity']
    ].sort_values(['track_genre', 'popularity'], ascending=[True, False])
)


## 9. Describe del Spotify ya limpio

In [ ]:
display(spotify_clean.describe(include='all').T)

## 10. Cleaning de Grammys paso a paso

In [9]:
grammy_work = grammy_df.copy()

text_cols = ['title', 'published_at', 'updated_at', 'category', 'nominee', 'artist', 'workers', 'img']
for col in text_cols:
    grammy_work[col] = grammy_work[col].fillna('').astype(str).str.strip()

grammy_work['year'] = pd.to_numeric(grammy_work['year'], errors='coerce').astype('Int64')
grammy_work['winner'] = (
    grammy_work['winner']
    .astype(str)
    .str.lower()
    .map({'true': 1, 'false': 0})
    .fillna(0)
    .astype(int)
)

grammy_work['artist_normalized'] = grammy_work['artist'].apply(normalize_text)
grammy_work['nominee_normalized'] = grammy_work['nominee'].apply(normalize_text)
grammy_work['category_normalized'] = grammy_work['category'].apply(normalize_text)

grammy_metrics = pd.DataFrame([
    {'metric': 'raw_rows', 'value': len(grammy_df)},
    {'metric': 'rows_with_blank_artist', 'value': int((grammy_work['artist'] == '').sum())},
    {'metric': 'distinct_categories', 'value': int(grammy_work['category'].nunique())},
    {'metric': 'distinct_years', 'value': int(grammy_work['year'].nunique())},
    {'metric': 'winner_values_observed', 'value': int(grammy_work['winner'].nunique())},
])

grammy_clean = grammy_work.copy()

display(grammy_metrics)
display(grammy_clean.head())

,metric,value
0,raw_rows,4810
1,rows_with_blank_artist,1840
2,distinct_categories,638
3,distinct_years,62
4,winner_values_observed,1


,year,title,published_at,updated_at,category,nominee,artist,workers,img,winner,artist_normalized,nominee_normalized,category_normalized
0,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,Bad Guy,Billie Eilish,"Finneas O'Connell, producer; Rob Kinelski & Finneas O'Connell, engineers/mixers; John Greenham, mastering engineer",https://www.grammy.com/sites/com/files/styles/artist_circle/public/muzooka/Billie%2BEilish/Billie%2520Eilish_1_1_159...,1,billie eilish,bad guy,record of the year
1,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,"Hey, Ma",Bon Iver,"BJ Burton, Brad Cook, Chris Messina & Justin Vernon, producers; BJ Burton, Zach Hanson & Chris Messina, engineers/mi...",https://www.grammy.com/sites/com/files/styles/artist_circle/public/muzooka/Bon%2BIver/Bon%2520Iver_1_1_1578385181.jp...,1,bon iver,hey ma,record of the year
2,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,7 rings,Ariana Grande,"Charles Anderson, Tommy Brown, Michael Foster & Victoria Monet, producers; Serban Ghenea, John Hanes, Billy Hickey &...",https://www.grammy.com/sites/com/files/styles/artist_circle/public/muzooka/Ariana%2BGrande/Ariana%2520Grande_1_1_157...,1,ariana grande,7 rings,record of the year
3,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,Hard Place,H.E.R.,"Rodney “Darkchild” Jerkins, producer; Joseph Hurtado, Jaycen Joshua, Derek Keota & Miki Tsutsumi, engineers/mixers; ...",https://www.grammy.com/sites/com/files/styles/artist_circle/public/muzooka/H.E.R./H.E.R._1_1_1594631035.jpg?itok=ClJ...,1,h e r,hard place,record of the year
4,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,Talk,Khalid,"Disclosure & Denis Kosiak, producers; Ingmar Carlson, Jon Castelli, Josh Deguzman, John Kercy, Denis Kosiak, Guy Law...",https://www.grammy.com/sites/com/files/styles/artist_circle/public/muzooka/Khalid/Khalid_1_1_1594578772.jpg?itok=2Hx...,1,khalid,talk,record of the year


## 11. Describe del Grammy ya limpio

In [10]:
display(grammy_clean.describe(include='all').T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
year,4810.0,<NA>,<NA>,<NA>,1995.566944,17.14972,1958.0,1983.0,1998.0,2010.0,2019.0
title,4810,62,62nd Annual GRAMMY Awards (2019),433,NaN,NaN,NaN,NaN,NaN,NaN,NaN
published_at,4810,4,2017-11-28T00:03:45-08:00,4205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
updated_at,4810,10,2019-09-10T01:08:19-07:00,778,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,4810,638,Song Of The Year,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nominee,4810,4132,Bridge Over Troubled Water,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
artist,4810,1659,,1840,NaN,NaN,NaN,NaN,NaN,NaN,NaN
workers,4810,2367,,2190,NaN,NaN,NaN,NaN,NaN,NaN,NaN
img,4810,1464,,1367,NaN,NaN,NaN,NaN,NaN,NaN,NaN
winner,4810.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0


## 12. Exploración del matching entre Spotify y Grammys

No existe una llave compartida entre ambas fuentes. Por eso el cruce más confiable usa `primary_artist_normalized + track_name_normalized` en Spotify contra `artist_normalized + nominee_normalized` en Grammys.

Los géneros se preservan para análisis posterior, pero no ayudan a conectar las dos fuentes. Cuando no hay match exacto por track, todavía se puede enriquecer a nivel de artista usando el histórico Grammy del `primary_artist`.


In [11]:
exact_matches = spotify_clean.merge(
    grammy_clean,
    left_on=['primary_artist_normalized', 'track_name_normalized'],
    right_on=['artist_normalized', 'nominee_normalized'],
    how='inner',
    suffixes=('_spotify', '_grammy')
)

artist_level_stats = (
    grammy_clean[grammy_clean['artist_normalized'] != '']
    .groupby(['artist_normalized', 'artist'], as_index=False)
    .agg(
        artist_grammy_win_count=('winner', 'sum'),
        artist_distinct_award_categories=('category', 'nunique'),
        artist_first_grammy_year=('year', 'min'),
        artist_last_grammy_year=('year', 'max')
    )
)

spotify_enriched = spotify_clean.merge(
    artist_level_stats,
    left_on='primary_artist_normalized',
    right_on='artist_normalized',
    how='left'
)

exact_track_ids = set(exact_matches['track_id'])
spotify_enriched['has_track_grammy_match'] = spotify_enriched['track_id'].isin(exact_track_ids).astype(int)
spotify_enriched['artist_grammy_win_count'] = spotify_enriched['artist_grammy_win_count'].fillna(0).astype(int)
spotify_enriched['track_match_type'] = 'no_match'
spotify_enriched.loc[spotify_enriched['artist_grammy_win_count'] > 0, 'track_match_type'] = 'artist_only'
spotify_enriched.loc[spotify_enriched['has_track_grammy_match'] == 1, 'track_match_type'] = 'track_and_artist'

match_summary = pd.DataFrame([
    {'metric': 'spotify_tracks_after_cleaning', 'value': len(spotify_clean)},
    {'metric': 'exact_track_and_artist_matches', 'value': int(spotify_enriched['has_track_grammy_match'].sum())},
    {'metric': 'artist_only_matches', 'value': int((spotify_enriched['track_match_type'] == 'artist_only').sum())},
    {'metric': 'unmatched_tracks', 'value': int((spotify_enriched['track_match_type'] == 'no_match').sum())},
])

display(match_summary)
display(spotify_enriched['track_match_type'].value_counts().to_frame('count'))

,metric,value
0,spotify_tracks_after_cleaning,89740
1,exact_track_and_artist_matches,769
2,artist_only_matches,5675
3,unmatched_tracks,83312


,count
track_match_type,
no_match,83312
artist_only,5675
track_and_artist,769


## 13. Ejemplos de matches exactos

Aquí puedes ver pistas concretas de por qué la lógica de normalización funciona.

In [12]:
exact_match_preview = exact_matches[[
    'track_id', 'primary_artist', 'track_name', 'category', 'year', 'nominee'
]].drop_duplicates().sort_values(['year', 'primary_artist', 'track_name'], ascending=[False, True, True])

display(exact_match_preview.head(30))

,track_id,primary_artist,track_name,category,year,nominee
2947,6ocbgoVGwYJhOv1GgI9NsF,Ariana Grande,7 rings,Record Of The Year,2019,7 rings
2948,6ocbgoVGwYJhOv1GgI9NsF,Ariana Grande,7 rings,Best Pop Solo Performance,2019,7 rings
1480,3e9HZxeyfWwjeyPAMmWSSQ,Ariana Grande,"thank u, next",Album Of The Year,2019,"thank u, next"
1481,3e9HZxeyfWwjeyPAMmWSSQ,Ariana Grande,"thank u, next",Best Pop Vocal Album,2019,"thank u, next"
456,154lbtHSyyOt0ao2QwbE4n,Billie Eilish,bad guy,Record Of The Year,2019,Bad Guy
457,154lbtHSyyOt0ao2QwbE4n,Billie Eilish,bad guy,Best Pop Solo Performance,2019,Bad Guy
678,1c8h1SBYY44ys4EuaIZJMn,Billie Eilish,bad guy,Record Of The Year,2019,Bad Guy
679,1c8h1SBYY44ys4EuaIZJMn,Billie Eilish,bad guy,Best Pop Solo Performance,2019,Bad Guy
948,2Fxmhks0bxGSBdJ92vM42m,Billie Eilish,bad guy,Record Of The Year,2019,Bad Guy
949,2Fxmhks0bxGSBdJ92vM42m,Billie Eilish,bad guy,Best Pop Solo Performance,2019,Bad Guy


## 14. Preguntas analíticas propuestas

Con este pipeline y este modelo, las preguntas más defendibles para el workshop serían:

- ¿La popularidad de Spotify parece reflejarse en los tracks que logran match con el histórico Grammy?
- ¿Variables como `danceability`, `energy`, `valence` o `instrumentalness` muestran patrones distintos en tracks con historial Grammy?
- ¿Qué perfil tienen los artistas con más apariciones o más nominaciones visibles en el dataset de premios?
- ¿Qué géneros o combinaciones de géneros aparecen con más frecuencia entre los tracks conectados con Grammys?

Limitación importante: con el archivo actual de Grammys no vemos claramente perdedores, porque el campo `winner` se comporta como si todo fuera histórico de ganadores. Entonces la pregunta de “artistas que más pierden” solo la podremos responder si en la revisión del dataset aparece otra evidencia o una fuente complementaria.


## 15. Conclusión

Este notebook deja visible el razonamiento detrás del pipeline:

- el profiling mostró por qué había que construir un track canónico sin perder los géneros de Spotify,
- la revisión de duplicados justificó consolidar por `track_id` y no deduplicar por nombre,
- el análisis de Grammys mostró limitaciones y comportamiento real del dataset,
- y la exploración de matching justificó usar un join exacto por track más enriquecimiento a nivel artista.

Con esto ya queda lista la base documental para seguir con la siguiente fase: decidir qué métricas finales queremos contestar en el dashboard y ajustar el ETL si hace falta.
